# Leakage Audit & Selection Bias Analysis

**Credit Risk Decision System - Phase 2 Ingestion Gate**

### Objectives:
1. **Zero-Leakage Verification (ADR 3.1):** Formally verify that no post-origination columns (`recoveries`, `collection_recovery_fee`, `last_pymnt_d`, `last_pymnt_amnt`, etc.) exist in training or test sets.
2. **Selection Bias & Reject Inference Audit:** Compare the population characteristics of applicants who were accepted vs rejected by Lending Club to quantify truncation and survivorship bias.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from config.constants import POST_ORIGINATION_COLS, IDENTIFIER_COLS, RAW_TARGET_COL, TARGET_COL
sns.set_theme(style='whitegrid')


## 1. Zero-Leakage Gate Audit

We inspect the schemas of all generated Parquet partitions (`train.parquet`, `val.parquet`, `test.parquet`, `stress_test.parquet`).
Any presence of post-origination data is an automatic failure.


In [ ]:
from scripts.audit_selection_bias import audit_leakage
audit_results = audit_leakage(PROJECT_ROOT / 'data' / 'processed')
for part, res in audit_results.items():
    print(f"{part:20s}: Status = {res['status']} | Rows = {res['row_count']:,} | Cols = {res['column_count']}")
    assert res['status'] == 'PASS', f"Leakage detected in {part}!"


## 2. Selection Bias Audit (Accepted vs Rejected Cohorts)

Because credit scoring models are trained exclusively on accepted borrowers whose terminal performance is known, the training cohort suffers from significant selection bias. Below we quantify the structural difference.


In [ ]:
from scripts.audit_selection_bias import audit_selection_bias
summary = audit_selection_bias(
    rejected_path=PROJECT_ROOT / 'data' / 'raw' / 'rejected_2007_to_2018Q4.csv.gz',
    train_parquet_path=PROJECT_ROOT / 'data' / 'processed' / 'train.parquet',
    sample_size=100000,
)
with open(PROJECT_ROOT / 'reports' / 'selection_bias_summary.json', 'r') as f:
    stats = json.load(f)['summary_statistics']
pd.DataFrame(stats).T


In [ ]:
from IPython.display import Image, display
fig_file = PROJECT_ROOT / 'reports' / 'figures' / 'selection_bias_comparison.png'
display(Image(filename=str(fig_file)))


## 3. Findings & Methodological Implications

- **Credit Score Divergence:** Accepted applicants have a median FICO of ~698, with virtually no applicants below 660. In contrast, rejected applicants have a median Risk Score of ~627, extending down below 500.
- **DTI Truncation:** Accepted loans cluster around a median DTI of ~18%, while rejected applicants have substantial mass above the 43% Qualified Mortgage cap.
- **Policy Implication:** Models trained strictly on accepted loans cannot be blindly extrapolated to deep subprime applications. A rigorous three-tier policy gate (Approve / Review / Reject) is vital to route borderline applicants to underwriter scrutiny.
